# Cvičení 3: Čtení údajů z občanského průkazu

Cílem cvičení je strojově přečíst fotografii obličeje, jméno a příjmení z fotografie občanského průkazu ČR. Průkaz může být vůči fotoaparátu libovolně natočený. Na fotografii jej najdeme pomocí **lokálních příznaků** a **homografie** vůči vzoru průkazu, narovnáme jej, vyřízneme zájmové oblasti a text přečteme pomocí **OCR** (optical character recognition). Postup nejprve vyvineme krok za krokem na jedné fotografii, pak jej převedeme do jediné funkce a použijeme na další fotografie.

1. Detekce průkazu
2. Rektifikace průkazu
3. Výřez fotografie obličeje, jména a příjmení
4. Čtení textu (OCR)
5. Funkce `ocr_cze_id` a její použití na další fotografie

Obecné pokyny:
- Můžete používat [OpenCV](https://docs.opencv.org/) nebo např. [scikit-image](https://scikit-image.org/). Text čtěte knihovnou [EasyOCR](https://github.com/JaidedAI/EasyOCR).
- Výsledky vykreslujte pomocí [matplotlib](https://matplotlib.org/).
- Svůj kód pište na místa označená komentářem `# TODO: implement`.
- Každá sekce obsahuje vzorový výstup. Váš výsledek nemusí být na pixel stejný, ale měl by vypadat podobně.
- Za každým bodem je buňka s testy ze souboru `tests/test_id_recognition.py`.
- Data pocházejí z datasetu [MIDV-500](https://arxiv.org/abs/1807.05786) a jsou v adresáři `data/10_cze_id/`: `10_cze_id_template.png` je **vzor** průkazu bez osobních údajů, `*.jpg` jsou **fotografie** průkazu pořízené v různých podmínkách.

In [ ]:
import sys

import cv2 as cv
import easyocr
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray

if ".." not in sys.path:
    sys.path.append("..")
from tests import test_id_recognition

## Načtení obrázků

Vzor průkazu a fotografie, na které budeme postup vyvíjet.

In [ ]:
template_rgb = cv.cvtColor(cv.imread("../data/10_cze_id/10_cze_id_template.png"), cv.COLOR_BGR2RGB)
scene_rgb = cv.cvtColor(cv.imread("../data/10_cze_id/KA10_01.jpg"), cv.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [2, 1]})
axes[0].imshow(template_rgb)
axes[0].set_title(f"template {template_rgb.shape[1]}x{template_rgb.shape[0]}")
axes[1].imshow(scene_rgb)
axes[1].set_title(f"scene {scene_rgb.shape[1]}x{scene_rgb.shape[0]}")
fig.tight_layout()

## 1. Detekce průkazu

- Najděte průkaz na fotografii `scene_rgb` a jeho rohy uložte do proměnné `corners`: pole tvaru `(4, 2)` se souřadnicemi `(x, y)` v pixelech fotografie v pořadí levý horní, pravý horní, pravý dolní a levý dolní roh.
- Doporučený postup:
  - zájmové body a jejich deskriptory ve vzoru i ve fotografii, např. SIFT (`cv.SIFT_create`),
  - párování deskriptorů (`cv.BFMatcher`) s ratio testem,
  - homografie mezi vzorem a fotografií odhadnutá robustně, např. `cv.findHomography` s metodou RANSAC,
  - převod rohů vzoru do fotografie (`cv.perspectiveTransform`).
- Vzor je mnohem menší než průkaz na fotografii, pomoci může jeho zvětšení.
- Vykreslete spárované body odpovídající homografii a nalezený čtyřúhelník.
- Test zkontroluje pořadí rohů a porovná `corners` se skutečnými pozicemi rohů vypočtením IoU čtyřúhelníků (alespoň **0.9**).

Vzorový výstup:

<figure class="image">
  <img src="../figures/id_recognition-expected_outputs_detection.png" alt="Vzorový výstup" style="width: 60%"/>
</figure>

In [ ]:
########################################
# TODO: implement

# corners = ...

# Vykresleni
# ...

########################################

In [ ]:
test_id_recognition.TestDetection.eval(corners=corners)

## 2. Rektifikace průkazu

- Narovnejte průkaz z fotografie do obrázku `card` o rozměru `CARD_SIZE` = 669×420 px (šířka × výška) tak, aby velikostí odpovídal vzoru `data/10_cze_id/10_cze_id.jpg`.
- Test porovná `card` s průkazem narovnaným podle anotovaných rohů pomocí [SSIM](https://en.wikipedia.org/wiki/Structural_similarity_index_measure) (alespoň **0,6**). Porovnávají se zmenšené šedotónové obrázky, takže posun o pár pixelů nevadí.

Vzorový výstup:

<figure class="image">
  <img src="../figures/id_recognition-expected_outputs_rectification.png" alt="Vzorový výstup" style="width: 50%"/>
</figure>

In [ ]:
CARD_SIZE = (669, 420)

########################################
# TODO: implement

# card = ...

# Vykresleni
# ...

########################################

In [ ]:
test_id_recognition.TestRectification.eval(card=card)

## 3. Výřez fotografie obličeje, jména a příjmení

- V narovnaném průkazu `card` určete obdélníky fotografie obličeje, příjmení a jména a výřezy uložte do slovníku `rois` (Region of Interest, ROI) s klíči `"photo"`, `"surname"` a `"name"`.
- Souřadnice odečtěte ručně z obrázku `data/10_cze_id/10_cze_id.jpg`. Výřez textu by měl obsahovat celý text s malou rezervou (i pro delší jména), ale ne sousední řádek.
- Výřezy nezvětšujte ani jinak neupravujte, předzpracování pro OCR patří do bodu 4.
- Test hledá v každém výřezu odpovídající část referenčního průkazu (normalizovaná korelace, `cv.matchTemplate`, alespoň **0.8**). Výřez musí obsahovat celou fotografii, resp. celé příjmení nebo jméno. Výřez textu nesmí obsahovat i druhý řádek a výřez fotografie nesmí být výrazně větší než fotografie.

Vzorový výstup:

<figure class="image">
  <img src="../figures/id_recognition-expected_outputs_crop.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement

# rois = ...

# Vykresleni
# ...

########################################

In [ ]:
test_id_recognition.TestROIs.eval(rois=rois)

## 4. Čtení textu (OCR)

- Implementujte funkci `ocr(roi)`, která pomocí EasyOCR přečte text z výřezu a vrátí jej jako řetězec.
  - Objekt `reader = easyocr.Reader(["cs"])` je vytvořený v buňce níže. Při prvním spuštění stáhne modely (asi 100 MB), bez grafické karty běží na procesoru.
  - Přesnost zlepší předzpracování výřezu, např. zvětšení, úprava kontrastu nebo doostření.
- Funkci použijte na výřezy jména a příjmení z bodu 3 a výsledek vykreslete.
- Test použije funkci na referenční výřezy jména a příjmení z této fotografie a na výřezy průkazů s jinými jmény. Velikost písmen ani mezery na okrajích nehrají roli. Každý přečtený text se smí od správného lišit nejvýše v **jednom znaku** (počítá se pomocí `difflib`). Bonusový test vyžaduje přesné přečtení všech textů.

Vzorový výstup:

<figure class="image">
  <img src="../figures/id_recognition-expected_outputs_ocr.png" alt="Vzorový výstup" style="width: 30%"/>
</figure>

In [ ]:
reader = easyocr.Reader(["cs"])

In [ ]:
def ocr(roi: NDArray[np.uint8]) -> str:
    """
    Read the text in the ROI.

    Args:
        roi: Image of the text in RGB order of dtype uint8.
    Returns:
        Text in the ROI.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# name = ...
# surname = ...

# Vykresleni
# ...

########################################

In [ ]:
test_id_recognition.TestOCR.eval(ocr_fn=ocr)

In [ ]:
test_id_recognition.TestOCRExact.eval(ocr_fn=ocr)

## 5. Funkce `ocr_cze_id` a její použití na další fotografie

- Převeďte celý postup do jediné funkce `ocr_cze_id(rgb)`, která z fotografie průkazu vrátí trojici `(photo, name, surname)`: výřez fotografie obličeje v RGB (z narovnaného průkazu velikosti `CARD_SIZE`, bez zvětšení), jméno a příjmení jako řetězce.
- Funkci použijte na fotografie ze seznamu `IMAGE_NAMES` a výsledky vykreslete.
- Testy použijí funkci na tytéž fotografie a na dvě fotografie, ve kterých je průkaz nahrazen průkazem s jiným jménem:
  - výřez fotografie obličeje musí na každé fotografii projít stejnou kontrolou jako v bodě 3,
  - alespoň **80 %** jmen a příjmení se smí lišit nejvýše v jednom znaku,
  - bonusový test vyžaduje přesné přečtení všech jmen a příjmení.
- Na kterých fotografiích OCR chybuje a proč?

Vzorový výstup:

<figure class="image">
  <img src="../figures/id_recognition-expected_outputs_ocr_cze_id.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def ocr_cze_id(rgb: NDArray[np.uint8]) -> tuple[NDArray[np.uint8], str, str]:
    """
    Read the photo of the face, the name and the surname from the image of the Czech ID card.

    Args:
        rgb: Image of the card in RGB order of dtype uint8.
    Returns:
        Photo of the face in RGB order cropped from the card rectified to CARD_SIZE, name and surname.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
IMAGE_NAMES = ["CA10_01", "HA10_06", "HS10_12", "KA10_01", "KS10_02", "PA10_30", "TA10_01", "TS10_01"]

########################################
# TODO: implement

# results = ...

# Vykresleni
# ...

########################################

In [ ]:
test_id_recognition.TestOcrCzeIdPhotos.eval(ocr_cze_id_fn=ocr_cze_id)

In [ ]:
test_id_recognition.TestOcrCzeIdTexts.eval(ocr_cze_id_fn=ocr_cze_id)

In [ ]:
test_id_recognition.TestOcrCzeIdExact.eval(ocr_cze_id_fn=ocr_cze_id)